# NB45 — Thermal Dynamics on Z$^*_{210}$

The character spectrum of NB44 gave us 48 labeled eigenvalues.
Now we study **dynamics**: how excitations propagate, thermalize, and mix
on the Cayley graph of $\mathbb{Z}^*_{210}$.

The character decomposition makes everything **analytically tractable**:

$$K(g,h,t) = \frac{1}{|G|}\sum_\chi \chi(g^{-1}h)\,e^{-t\lambda_\chi}$$

Five identities emerge:

| # | Identity | Domain |
|---|----------|--------|
| 51 | Heat trace factorization | Dynamics |
| 52 | Triangle-free Cayley graph | Geometry |
| 53 | Closed-form partition function | Thermodynamics |
| 54 | Spectral determinant $= 2^{25}\cdot 3^{16}\cdot 5^{13}\cdot 7^8$ | Algebra |
| 55 | $\mathbb{Q}(\sqrt{3})$ universality | Number theory |

## §1 Setup

Rebuild spectra from NB44: per-prime generators, separable and coupled
generating sets, character eigenvalue formula.

In [13]:
import sys, os, numpy as np
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
from solenoid_algebra import SA, PRIMES, P, PHI, PRIMORIALS
from numpy import pi, exp, sqrt, log
from collections import Counter

# Per-prime generators (CRT lifts of primitive roots)
per_prime_gens = {3: 71, 5: 127, 7: 31}

# Generating sets
gen_set_sep = sorted(set(
    g for p_g in per_prime_gens.values()
    for g in [p_g, SA.inverse(p_g)]
))

coupled_raw = [17, 23, 37]
gen_set_coupled = sorted(set(
    g for r in coupled_raw for g in [r, SA.inverse(r)]
))

# Character eigenvalue formula
all_chars = SA.all_character_indices()

def char_eigenvalue(cidx, gen_set):
    return sum(1.0 - SA.character(cidx, s).real for s in gen_set)

# Per-prime spectra
per_prime_spectra = {}
for p, g in per_prime_gens.items():
    gens_p = sorted(set([g, SA.inverse(g)]))
    per_prime_spectra[p] = [char_eigenvalue(cidx, gens_p)
                            for cidx in all_chars
                            if all(cidx[j] == 0 for j in range(len(PRIMES))
                                   if PRIMES[j] != p and PRIMES[j] != 2)]

# Full spectra
sep_spectrum = [(cidx, char_eigenvalue(cidx, gen_set_sep)) for cidx in all_chars]
sep_spectrum.sort(key=lambda x: x[1])
coup_spectrum = [(cidx, char_eigenvalue(cidx, gen_set_coupled)) for cidx in all_chars]
coup_spectrum.sort(key=lambda x: x[1])

# Degeneracy structure (separable)
sep_rounded = [round(lam) for _, lam in sep_spectrum]
deg_counter = Counter(sep_rounded)
distinct_levels = sorted(deg_counter.keys())
degeneracies = [deg_counter[k] for k in distinct_levels]

print(f"Z*_{P}: {PHI} elements, lambda(P) = 12")
print(f"Separable |S| = {len(gen_set_sep)}: {gen_set_sep}")
print(f"Coupled   |S| = {len(gen_set_coupled)}: {gen_set_coupled}")
print(f"Levels: {distinct_levels}")
print(f"Degeneracies: {degeneracies}")

Z*_210: 48 elements, lambda(P) = 12
Separable |S| = 5: [31, 43, 61, 71, 127]
Coupled   |S| = 6: [17, 23, 37, 137, 173, 193]
Levels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Degeneracies: [1, 2, 3, 8, 4, 12, 4, 8, 3, 2, 1]


## §2 Identity 51 — Heat Trace Factorization

Under separable generators, characters decompose as tensor products:
$\chi = \chi_3 \otimes \chi_5 \otimes \chi_7$ with
$\lambda_\chi = \lambda_{a_3} + \lambda_{a_5} + \lambda_{a_7}$.

The heat trace $\Theta(t) = \sum_\chi e^{-t\lambda_\chi}$ therefore **factorizes**:

$$\boxed{\Theta_{\rm sep}(t) = \Theta_3(t)\cdot\Theta_5(t)\cdot\Theta_7(t)}$$

where $\Theta_p(t) = \sum_{a=0}^{\varphi(p)-1} e^{-t\lambda_p(a)}$.

The return probability $P_{\rm return}(t) = \Theta(t)/|G|$ inherits the product structure:
each faculty diffuses **independently**.

In [3]:
# Per-prime heat traces
def theta_p(p, t):
    return sum(exp(-t * lam) for lam in per_prime_spectra[p])

def theta_sep(t):
    return sum(d * exp(-t * k) for k, d in zip(distinct_levels, degeneracies))

def theta_coup(t):
    return sum(exp(-t * lam) for _, lam in coup_spectrum)

# Verify factorization
ts = np.linspace(0.01, 10, 200)
max_err = 0
for t in ts:
    direct = theta_sep(t)
    product = theta_p(3, t) * theta_p(5, t) * theta_p(7, t)
    max_err = max(max_err, abs(direct - product))

print("Heat trace factorization: Theta_sep(t) = Theta_3 * Theta_5 * Theta_7")
print(f"  Max error over t in [0.01, 10]: {max_err:.2e}")
print(f"  EXACT: {max_err < 1e-13}")

# Sample values
print(f"\n{'t':>6} {'Theta_direct':>14} {'Product':>14} {'Diff':>10}")
for t in [0.1, 0.5, 1.0, 2.0, 5.0]:
    d = theta_sep(t)
    p = theta_p(3, t) * theta_p(5, t) * theta_p(7, t)
    print(f"{t:6.2f} {d:14.8f} {p:14.8f} {abs(d-p):10.2e}")

# Return probability comparison
print(f"\nReturn probability P(t) = Theta(t)/|G|:")
print(f"{'t':>6} {'P_sep':>12} {'P_coup':>12} {'P_uniform':>12}")
for t in [0, 0.5, 1.0, 3.0, 10.0]:
    ps = theta_sep(t) / PHI
    pc = theta_coup(t) / PHI
    print(f"{t:6.1f} {ps:12.6f} {pc:12.6f} {1/PHI:12.6f}")

Heat trace factorization: Theta_sep(t) = Theta_3 * Theta_5 * Theta_7
  Max error over t in [0.01, 10]: 7.11e-15
  EXACT: True

     t   Theta_direct        Product       Diff
  0.10    29.84899559    29.84899559   7.11e-15
  0.50     7.15273224     7.15273224   8.88e-16
  1.00     2.71268787     2.71268787   0.00e+00
  2.00     1.34736575     1.34736575   4.44e-16
  5.00     1.01361455     1.01361455   0.00e+00

Return probability P(t) = Theta(t)/|G|:
     t        P_sep       P_coup    P_uniform
   0.0     1.000000     1.000000     0.020833
   0.5     0.149015     0.108025     0.020833
   1.0     0.056514     0.045905     0.020833
   3.0     0.023084     0.024576     0.020833
  10.0     0.020835     0.020847     0.020833


## §3 Identity 52 — Triangle-Free Cayley Graph

For the Laplacian $L = |S|I - A$ on a regular graph:

$$\operatorname{Tr}(L^3) = |G|\cdot|S|^3 + 3|G|\cdot|S|^2 - \operatorname{Tr}(A^3)$$

$\operatorname{Tr}(A^3)$ counts ordered triples $(s_1,s_2,s_3)\in S^3$ with $s_1 s_2 s_3 = e$.

$$\boxed{\operatorname{Tr}(A^3) = 0 \text{ for both generator sets}}$$

No three generators multiply to the identity. The Cayley graph is **triangle-free** (girth $\geq 4$).

Since $\mathbb{Z}^*_{210}$ is abelian, any two generators commissioning a commutator give $[s_1,s_2]=e$,
so 4-cycles exist whenever $|S| \geq 2$. The girth is exactly 4.

In [4]:
# Verify Tr(A^3) = 0 combinatorially
def count_triangles(gen_set):
    count = 0
    for s1 in gen_set:
        for s2 in gen_set:
            for s3 in gen_set:
                if (s1 * s2 * s3) % P == 1:
                    count += 1
    return count

tri_sep = count_triangles(gen_set_sep)
tri_coup = count_triangles(gen_set_coupled)

print("Triangle count: #{(s1,s2,s3) in S^3 : s1*s2*s3 = 1}")
print(f"  Separable: {tri_sep}")
print(f"  Coupled:   {tri_coup}")

# Verify via spectral moments
tr_L3 = sum(k**3 * d for k, d in zip(distinct_levels, degeneracies))
predicted = PHI * len(gen_set_sep)**3 + 3 * PHI * len(gen_set_sep)**2
tr_A3_from_spectrum = predicted - tr_L3

print(f"\nSpectral verification (separable):")
print(f"  Tr(L^3) = sum(k^3 * d_k) = {tr_L3}")
print(f"  |G|*|S|^3 + 3|G|*|S|^2 = {predicted}")
print(f"  Tr(A^3) = {tr_A3_from_spectrum}")

# Girth = 4: show an explicit 4-cycle
# 71 is self-inverse (71^2 = 1 mod 210), and 31*61 = 1 mod 210
cycle = [71, 31, 71, 61]
product = 1
for s in cycle:
    product = (product * s) % P
print(f"\nExplicit 4-cycle: {cycle}")
print(f"  Product mod {P} = {product} (should be 1)")
print(f"  Girth = 4")

Triangle count: #{(s1,s2,s3) in S^3 : s1*s2*s3 = 1}
  Separable: 0
  Coupled:   0

Spectral verification (separable):
  Tr(L^3) = sum(k^3 * d_k) = 9600
  |G|*|S|^3 + 3|G|*|S|^2 = 9600
  Tr(A^3) = 0

Explicit 4-cycle: [71, 31, 71, 61]
  Product mod 210 = 1 (should be 1)
  Girth = 4


## §4 Identity 53 — Closed-Form Partition Function

The partition function $Z(\beta) = \sum_\chi e^{-\beta\lambda_\chi} = \Theta(\beta)$.

Since $Z_3(\beta) = 1 + e^{-2\beta}$ and $Z_5(\beta) = (1+e^{-2\beta})^2$ (a perfect square!),
the primes 3 and 5 combine into a **cube**:

$$Z_3 \cdot Z_5 = (1 + e^{-2\beta})^3$$

The prime-7 factor $Z_7 = 1 + 2e^{-\beta} + 2e^{-3\beta} + e^{-4\beta}$ is a palindromic polynomial
in $x = e^{-\beta}$ that factors over $\mathbb{Q}(\sqrt{3})$:

$$Z_7(x) = (x^2 + (1+\sqrt{3})x + 1)(x^2 + (1-\sqrt{3})x + 1)$$

The full partition function:

$$\boxed{Z_{\rm sep}(\beta) = (1 + e^{-2\beta})^3 \cdot (e^{-2\beta}+(1{+}\sqrt{3})e^{-\beta}+1)
\cdot(e^{-2\beta}+(1{-}\sqrt{3})e^{-\beta}+1)}$$

In [5]:
s3 = sqrt(3)

# Verify per-prime partition functions
print("Per-prime partition functions:")
print(f"  Z*_3 spectrum: {per_prime_spectra[3]}")
print(f"  Z*_5 spectrum: {per_prime_spectra[5]}")
print(f"  Z*_7 spectrum: {per_prime_spectra[7]}")

# Z5 = (1 + e^{-2beta})^2 ?
betas = [0.1, 0.5, 1.0, 2.0, 5.0]
print(f"\nZ5 = (1 + e^{{-2beta}})^2:")
z5_match = True
for beta in betas:
    Z5 = theta_p(5, beta)
    Z5_sq = (1 + exp(-2*beta))**2
    ok = abs(Z5 - Z5_sq) < 1e-14
    z5_match &= ok
    print(f"  beta={beta}: Z5={Z5:.10f}, (1+e^-2b)^2={Z5_sq:.10f}, ok={ok}")

# Full closed form
print(f"\nFull closed form Z_sep = (1+x^2)^3 * f1 * f2:")
max_closed_err = 0
for beta in np.linspace(0.01, 20, 500):
    x = exp(-beta)
    Z_direct = theta_sep(beta)
    f1 = x**2 + (1 + s3) * x + 1
    f2 = x**2 + (1 - s3) * x + 1
    Z_closed = (1 + x**2)**3 * f1 * f2
    max_closed_err = max(max_closed_err, abs(Z_direct - Z_closed))

print(f"  Max error over beta in [0.01, 20]: {max_closed_err:.2e}")
print(f"  EXACT: {max_closed_err < 1e-13}")

# Z7 polynomial roots
print(f"\nZ7 palindromic polynomial: x^4 + 2x^3 + 2x + 1")
print(f"  Factors: (x^2 + (1+sqrt3)x + 1)(x^2 + (1-sqrt3)x + 1)")
print(f"  sqrt(3) enters through y^2 + 2y - 2 = 0 with y = x + 1/x")

Per-prime partition functions:
  Z*_3 spectrum: [np.float64(0.0), np.float64(2.0)]
  Z*_5 spectrum: [np.float64(0.0), np.float64(2.0), np.float64(4.0), np.float64(1.9999999999999996)]
  Z*_7 spectrum: [np.float64(0.0), np.float64(0.9999999999999998), np.float64(2.9999999999999996), np.float64(4.0), np.float64(3.000000000000001), np.float64(1.0000000000000009)]

Z5 = (1 + e^{-2beta})^2:
  beta=0.1: Z5=3.3077815522, (1+e^-2b)^2=3.3077815522, ok=True
  beta=0.5: Z5=1.8710941656, (1+e^-2b)^2=1.8710941656, ok=True
  beta=1.0: Z5=1.2889862054, (1+e^-2b)^2=1.2889862054, ok=True
  beta=2.0: Z5=1.0369667404, (1+e^-2b)^2=1.0369667404, ok=True
  beta=5.0: Z5=1.0000908019, (1+e^-2b)^2=1.0000908019, ok=True

Full closed form Z_sep = (1+x^2)^3 * f1 * f2:
  Max error over beta in [0.01, 20]: 1.42e-14
  EXACT: True

Z7 palindromic polynomial: x^4 + 2x^3 + 2x + 1
  Factors: (x^2 + (1+sqrt3)x + 1)(x^2 + (1-sqrt3)x + 1)
  sqrt(3) enters through y^2 + 2y - 2 = 0 with y = x + 1/x


## §5 Identity 54 — Spectral Determinant

The spectral determinant $\det'(L) = \prod_{\lambda>0}\lambda^{d_\lambda}$
encodes the complete spectral weight. By Kirchhoff's theorem, the number of
spanning trees is $\tau = \det'(L)/|G|$.

Since all non-zero eigenvalues are integers in $\{1,\ldots,10\}$, the prime
factorization of $\det'(L)$ involves only primes $\leq 10$. But in fact:

$$\boxed{\det'(L) = 2^{25}\cdot 3^{16}\cdot 5^{13}\cdot 7^8}$$

**Only the four primordial primes $\{2,3,5,7\}$ appear.**

Spanning trees: $\tau = 2^{21}\cdot 3^{15}\cdot 5^{13}\cdot 7^8$ (dividing by $|G|=48=2^4\cdot 3$).

In [6]:
# Exact integer arithmetic for spectral determinant
det_int = 1
print("Spectral determinant (exact integer arithmetic):")
for k, d in zip(distinct_levels[1:], degeneracies[1:]):
    term = k ** d
    det_int *= term
    print(f"  {k}^{d} = {term}")

print(f"\ndet'(L) = {det_int}")

# Prime factorization
n = det_int
factors = {}
for p in [2, 3, 5, 7, 11, 13]:
    count = 0
    while n % p == 0:
        n //= p
        count += 1
    if count > 0:
        factors[p] = count

print(f"Prime factorization: {factors}")
print(f"Remaining cofactor: {n}")

# Verify
predicted_det = 2**25 * 3**16 * 5**13 * 7**8
det_match = (det_int == predicted_det)
print(f"\n2^25 * 3^16 * 5^13 * 7^8 = {predicted_det}")
print(f"MATCH: {det_match}")

# Exponent accounting
print(f"\nExponent accounting:")
print(f"  2: from 2^3, 4^4=2^8, 6^4=2^4, 8^3=2^9, 10^1=2^1 => 3+8+4+9+1 = 25")
print(f"  3: from 3^8, 6^4=3^4, 9^2=3^4 => 8+4+4 = 16")
print(f"  5: from 5^12, 10^1=5^1 => 12+1 = 13")
print(f"  7: from 7^8 => 8")

# Kirchhoff spanning trees
tau = det_int // PHI
tau_predicted = 2**21 * 3**15 * 5**13 * 7**8
tau_match = (tau == tau_predicted)
print(f"\nSpanning trees = det'(L) / |G| = {tau}")
print(f"  = 2^21 * 3^15 * 5^13 * 7^8 = {tau_predicted}")
print(f"  MATCH: {tau_match}")

Spectral determinant (exact integer arithmetic):
  1^2 = 1
  2^3 = 8
  3^8 = 6561
  4^4 = 256
  5^12 = 244140625
  6^4 = 1296
  7^8 = 5764801
  8^3 = 512
  9^2 = 81
  10^1 = 10

det'(L) = 10164460759757660160000000000000
Prime factorization: {2: 25, 3: 16, 5: 13, 7: 8}
Remaining cofactor: 1

2^25 * 3^16 * 5^13 * 7^8 = 10164460759757660160000000000000
MATCH: True

Exponent accounting:
  2: from 2^3, 4^4=2^8, 6^4=2^4, 8^3=2^9, 10^1=2^1 => 3+8+4+9+1 = 25
  3: from 3^8, 6^4=3^4, 9^2=3^4 => 8+4+4 = 16
  5: from 5^12, 10^1=5^1 => 12+1 = 13
  7: from 7^8 => 8

Spanning trees = det'(L) / |G| = 211759599161617920000000000000
  = 2^21 * 3^15 * 5^13 * 7^8 = 211759599161617920000000000000
  MATCH: True


## §6 Identity 55 — $\mathbb{Q}(\sqrt{3})$ Universality

Under separable generators, all eigenvalues are integers (Niven, Identity 46).
Under coupled generators, irrational eigenvalues appear — but which irrationals?

The Carmichael function $\lambda(210) = 12$ is the group exponent.
All characters factor through $\mathbb{Z}/12\mathbb{Z}$, so character values lie in
$\mathbb{Q}(\zeta_{12})$. The **real subfield** is:

$$\mathbb{Q}(\cos(2\pi/12)) = \mathbb{Q}(\cos(\pi/6)) = \mathbb{Q}(\sqrt{3})$$

Therefore **every eigenvalue** $\lambda_\chi \in \mathbb{Z}[\sqrt{3}]$.

$$\boxed{\text{All 48 coupled eigenvalues} \in \mathbb{Z}[\sqrt{3}]}$$

This is the same $\sqrt{3}$ that governs:
- The coupled spectral gap $6 - 3\sqrt{3}$ (Identity 49)
- The prime-7 partition function factorization (Identity 53)

In [7]:
# Decompose all coupled eigenvalues as a + b*sqrt(3)
s3 = sqrt(3)
integers = []
irrationals = []

for cidx in sorted(all_chars):
    lam = char_eigenvalue(cidx, gen_set_coupled)
    rounded = round(lam)
    if abs(lam - rounded) < 1e-10:
        integers.append((cidx, rounded))
    else:
        for a_try in range(13):
            residual = lam - a_try
            b_cand = residual / s3
            b_round = round(b_cand)
            if abs(b_cand - b_round) < 1e-10 and b_round != 0:
                irrationals.append((cidx, a_try, int(b_round)))
                break

print(f"Coupled eigenvalue classification:")
print(f"  Integer modes: {len(integers)}")
print(f"  Z[sqrt3] modes: {len(irrationals)}")
print(f"  Total: {len(integers) + len(irrationals)} = |G| = {PHI}")

all_in_zsqrt3 = (len(integers) + len(irrationals) == PHI)
print(f"  ALL in Z[sqrt(3)]: {all_in_zsqrt3}")

# Irrational families
ab_groups = Counter()
for _, a, b in irrationals:
    ab_groups[(a, b)] += 1
print(f"\nIrrational eigenvalue families:")
for (a, b), cnt in sorted(ab_groups.items()):
    val = a + b * s3
    print(f"  {a} + ({b})*sqrt(3) = {val:.6f}: {cnt} modes")

# Integer families
int_groups = Counter()
for _, v in integers:
    int_groups[v] += 1
print(f"\nInteger eigenvalue families:")
for v, cnt in sorted(int_groups.items()):
    print(f"  {v}: {cnt} modes")

# The field is determined by lambda(P) = 12
print(f"\nlambda({P}) = 12")
print(f"cos(2*pi/12) = cos(pi/6) = sqrt(3)/2 = {sqrt(3)/2:.10f}")
print(f"Real cyclotomic field: Q(sqrt(3))")

Coupled eigenvalue classification:
  Integer modes: 32
  Z[sqrt3] modes: 16
  Total: 48 = |G| = 48
  ALL in Z[sqrt(3)]: True

Irrational eigenvalue families:
  6 + (-3)*sqrt(3) = 0.803848: 2 modes
  6 + (-1)*sqrt(3) = 4.267949: 6 modes
  6 + (1)*sqrt(3) = 7.732051: 6 modes
  6 + (3)*sqrt(3) = 11.196152: 2 modes

Integer eigenvalue families:
  0: 1 modes
  3: 2 modes
  4: 3 modes
  5: 6 modes
  6: 8 modes
  7: 6 modes
  8: 3 modes
  9: 2 modes
  12: 1 modes

lambda(210) = 12
cos(2*pi/12) = cos(pi/6) = sqrt(3)/2 = 0.8660254038
Real cyclotomic field: Q(sqrt(3))


## §7 Thermodynamics

The partition function determines all thermodynamic quantities:

$$\langle E\rangle = -\frac{\partial\log Z}{\partial\beta}, \qquad
S = \log Z + \beta\langle E\rangle, \qquad
C = \beta^2\,\text{Var}(\lambda)$$

The **Schottky anomaly** — a peak in the specific heat — signals the temperature
regime where thermal fluctuations are maximized. Its location depends on
the full spectral structure, not just the gap.

In [8]:
# Thermodynamic functions
def thermo(spectrum_data, beta, use_degeneracy=False, levels=None, degs=None):
    # Compute Z, E, S, C from spectrum.
    if use_degeneracy:
        boltz = [d * exp(-beta * k) for k, d in zip(levels, degs)]
        lams = levels
        weights = degs
    else:
        lams = [l for _, l in spectrum_data]
        boltz = [exp(-beta * l) for l in lams]
        weights = [1] * len(lams)
    Z = sum(boltz)
    E = sum(l * w * exp(-beta * l) for l, w in zip(lams, weights)) / Z
    E2 = sum(l**2 * w * exp(-beta * l) for l, w in zip(lams, weights)) / Z
    C = beta**2 * (E2 - E**2)
    S = log(Z) + beta * E
    return Z, E, S, C

# Find Schottky peaks
betas_fine = np.linspace(0.01, 5.0, 2000)
Cs_sep = [thermo(None, b, True, distinct_levels, degeneracies)[3] for b in betas_fine]
Cs_coup = [thermo(coup_spectrum, b)[3] for b in betas_fine]

idx_s = np.argmax(Cs_sep)
idx_c = np.argmax(Cs_coup)
beta_peak_s = betas_fine[idx_s]
beta_peak_c = betas_fine[idx_c]

print("Schottky anomaly (specific heat peak):")
print(f"  Separable: beta* = {beta_peak_s:.4f}, C_max = {Cs_sep[idx_s]:.6f}")
print(f"  Coupled:   beta* = {beta_peak_c:.4f}, C_max = {Cs_coup[idx_c]:.6f}")
print(f"  Peak shift: beta_coup/beta_sep = {beta_peak_c/beta_peak_s:.4f}")

# Thermodynamic table
print(f"\n{'beta':>6} {'E_sep':>8} {'E_coup':>8} {'S_sep':>8} {'S_coup':>8} {'C_sep':>8} {'C_coup':>8}")
for beta in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    _, Es, Ss, Cs = thermo(None, beta, True, distinct_levels, degeneracies)
    _, Ec, Sc, Cc = thermo(coup_spectrum, beta)
    print(f"{beta:6.2f} {Es:8.4f} {Ec:8.4f} {Ss:8.4f} {Sc:8.4f} {Cs:8.4f} {Cc:8.4f}")

# High-T and low-T limits
print(f"\nHigh-T limit: S -> log(48) = {log(PHI):.4f}")
print(f"Low-T limit:  S -> 0, E -> 0")

Schottky anomaly (specific heat peak):
  Separable: beta* = 1.2581, C_max = 2.050779
  Coupled:   beta* = 0.7913, C_max = 2.216532
  Peak shift: beta_coup/beta_sep = 0.6290

  beta    E_sep   E_coup    S_sep   S_coup    C_sep   C_coup
  0.01   4.9500   5.9400   3.8710   3.8709   0.0005   0.0006
  0.10   4.5020   5.3971   3.8464   3.8410   0.0494   0.0609
  0.50   2.7205   2.8762   3.3277   3.0839   0.9451   1.4976
  1.00   1.3128   0.9019   2.3108   1.6919   1.9413   1.9657
  2.00   0.3328   0.2469   0.9637   0.8362   1.6508   0.7503
  5.00   0.0136   0.0279   0.0814   0.1747   0.3418   0.5410
 10.00   0.0001   0.0005   0.0010   0.0058   0.0091   0.0417

High-T limit: S -> log(48) = 3.8712
Low-T limit:  S -> 0, E -> 0


## §8 Cross-Verification

Direct matrix construction confirms all analytical results.

In [9]:
# Build adjacency and Laplacian matrices directly
idx_map = {g: i for i, g in enumerate(SA.Z_star)}

def build_adjacency(gen_set):
    A = np.zeros((PHI, PHI))
    for g in SA.Z_star:
        for s in gen_set:
            h = (g * s) % P
            A[idx_map[g], idx_map[h]] = 1.0
    return A

A_sep = build_adjacency(gen_set_sep)
A_coup = build_adjacency(gen_set_coupled)

# Tr(A^3) directly
trA3_sep = np.trace(A_sep @ A_sep @ A_sep)
trA3_coup = np.trace(A_coup @ A_coup @ A_coup)
print(f"Direct Tr(A^3) separable: {trA3_sep:.1f}")
print(f"Direct Tr(A^3) coupled:   {trA3_coup:.1f}")

# Eigenvalue comparison
L_sep = len(gen_set_sep) * np.eye(PHI) - A_sep
eigs_matrix = sorted(np.linalg.eigvalsh(L_sep))
eigs_char = sorted(lam for _, lam in sep_spectrum)
sep_diff = max(abs(a - b) for a, b in zip(eigs_matrix, eigs_char))
print(f"\nSeparable spectrum: matrix vs character diff = {sep_diff:.2e}")

L_coup = len(gen_set_coupled) * np.eye(PHI) - A_coup
eigs_matrix_c = sorted(np.linalg.eigvalsh(L_coup))
eigs_char_c = sorted(lam for _, lam in coup_spectrum)
coup_diff = max(abs(a - b) for a, b in zip(eigs_matrix_c, eigs_char_c))
print(f"Coupled spectrum:   matrix vs character diff = {coup_diff:.2e}")

print(f"\nBoth EXACT (< 1e-13): {sep_diff < 1e-13 and coup_diff < 1e-13}")

Direct Tr(A^3) separable: 0.0
Direct Tr(A^3) coupled:   0.0

Separable spectrum: matrix vs character diff = 7.11e-15
Coupled spectrum:   matrix vs character diff = 5.33e-15

Both EXACT (< 1e-13): True


## §8b Spectral Zeta Function (Frontier)

The **spectral zeta function** of the Cayley graph Laplacian is:

$$\zeta_L(s) = \sum_{\lambda > 0} d_\lambda \, \lambda^{-s}$$

where the sum runs over distinct nonzero eigenvalues with their degeneracies.
For integer spectrum (the separable case), this is exact. Special values
connect to quantities already computed:

| Value | Formula | Meaning |
|-------|---------|---------|
| $\zeta_L(0)$ | $\sum d_\lambda$ | Count of nonzero eigenvalues $= \|G\| - 1$ |
| $\zeta_L(-1)$ | $\sum d_\lambda \cdot \lambda$ | $\mathrm{Tr}(L) = \|G\| \cdot \|S\|$ |
| $\zeta_L'(0)$ | $-\sum d_\lambda \log\lambda$ | $-\log\det'(L)$ |

**Frontier observation**: $\zeta_L(-1) = 240 = |\Phi(E_8)|$. Whether this
connection to the E₈ root system is structural or arithmetically coincidental
remains an open question.

In [11]:
# Spectral zeta function for separable Laplacian
# Uses distinct_levels and degeneracies from §1 setup

def spectral_zeta(s, levels=distinct_levels, degs=degeneracies):
    # Sum over nonzero eigenvalues only (skip lambda=0)
    return sum(d * k**(-s) for k, d in zip(levels[1:], degs[1:]))

# Special values
zeta_0 = spectral_zeta(0)
zeta_neg1 = spectral_zeta(-1)
zeta_neg2 = spectral_zeta(-2)
zeta_1 = spectral_zeta(1)

print("Spectral zeta function zeta_L(s) = sum d_k * k^{-s}")
print("=" * 55)
print(f"  zeta(0)  = {zeta_0:.1f}   (|G|-1 = {PHI-1})")
print(f"  zeta(-1) = {zeta_neg1:.1f} (Tr(L) = |G|*|S| = {PHI}*{len(gen_set_sep)} = {PHI*len(gen_set_sep)})")
print(f"  zeta(-2) = {zeta_neg2:.1f}")
print(f"  zeta(1)  = {zeta_1:.8f}")
print()

# Verify zeta(0) = |G| - 1 = 47
print(f"zeta(0) == |G|-1 = 47:  {abs(zeta_0 - 47) < 1e-10}")

# Verify zeta(-1) = Tr(L) = 240
print(f"zeta(-1) == 240:        {abs(zeta_neg1 - 240) < 1e-10}")

# The 240 connection: Tr(L) = phi(210) * |S| = 48 * 5 = 240 = |Phi(E8)|
print(f"\n240 decomposition: phi(P4)*|S| = {PHI}*{len(gen_set_sep)} = {PHI*len(gen_set_sep)}")
print(f"|S| = {len(gen_set_sep)} because gen 71 is self-inverse (71^2 = 5041 = 24*210 + 1)")
print(f"So |S| = 2*3 - 1 = 5 (three generators, one self-inverse)")

# zeta'(0) = -log det'(L) (connects to Identity 54)
det_prime = 2**25 * 3**16 * 5**13 * 7**8
zeta_prime_0 = -sum(d * log(k) for k, d in zip(distinct_levels[1:], degeneracies[1:]))
log_det_from_54 = log(float(det_prime))
print(f"\nzeta'(0)      = {zeta_prime_0:.10f}")
print(f"-log det'(L)  = {-log_det_from_54:.10f}")
print(f"Match:          {abs(zeta_prime_0 - (-log_det_from_54)) < 1e-6}")

print("\n[FRONTIER] zeta(-1) = 240 = |Phi(E8)| — structural significance TBD")

Spectral zeta function zeta_L(s) = sum d_k * k^{-s}
  zeta(0)  = 47.0   (|G|-1 = 47)
  zeta(-1) = 240.0 (Tr(L) = |G|*|S| = 48*5 = 240)
  zeta(-2) = 1440.0
  zeta(1)  = 12.07341270

zeta(0) == |G|-1 = 47:  True
zeta(-1) == 240:        True

240 decomposition: phi(P4)*|S| = 48*5 = 240
|S| = 5 because gen 71 is self-inverse (71^2 = 5041 = 24*210 + 1)
So |S| = 2*3 - 1 = 5 (three generators, one self-inverse)

zeta'(0)      = -71.3964501868
-log det'(L)  = -71.3964501868
Match:          True

[FRONTIER] zeta(-1) = 240 = |Phi(E8)| — structural significance TBD


## §9 Scorecard

In [12]:
checks = {}

# Identity 51: Heat trace factorization
max_fact_err = 0
for t in np.linspace(0.01, 10, 200):
    direct = theta_sep(t)
    product = theta_p(3, t) * theta_p(5, t) * theta_p(7, t)
    max_fact_err = max(max_fact_err, abs(direct - product))
checks["51: Heat trace factorizes"] = max_fact_err < 1e-13

# Identity 52: Triangle-free
checks["52: Triangle-free (both)"] = (
    int(round(trA3_sep)) == 0 and int(round(trA3_coup)) == 0
)

# Identity 53: Closed-form partition function
max_closed_err = 0
for beta in np.linspace(0.01, 20, 500):
    x = exp(-beta)
    Z_direct = theta_sep(beta)
    f1 = x**2 + (1 + sqrt(3)) * x + 1
    f2 = x**2 + (1 - sqrt(3)) * x + 1
    Z_closed = (1 + x**2)**3 * f1 * f2
    max_closed_err = max(max_closed_err, abs(Z_direct - Z_closed))
checks["53: Z = (1+e^-2b)^3 * f1*f2"] = max_closed_err < 1e-13

# Identity 54: Spectral determinant
det_exact = 1
for k, d in zip(distinct_levels[1:], degeneracies[1:]):
    det_exact *= k ** d
checks["54: det'(L) = 2^25*3^16*5^13*7^8"] = (
    det_exact == 2**25 * 3**16 * 5**13 * 7**8
)

# Identity 55: Q(sqrt3) universality
n_accounted = 0
for cidx in all_chars:
    lam = char_eigenvalue(cidx, gen_set_coupled)
    r = round(lam)
    if abs(lam - r) < 1e-10:
        n_accounted += 1
    else:
        for a in range(13):
            b_cand = (lam - a) / sqrt(3)
            if abs(b_cand - round(b_cand)) < 1e-10:
                n_accounted += 1
                break
checks["55: All coupled eigs in Z[sqrt3]"] = (n_accounted == PHI)

print("=" * 60)
print("SCORECARD: Thermal Dynamics on Z*_210")
print("=" * 60)
n_pass = 0
for name, ok in checks.items():
    tag = "PASS" if ok else "FAIL"
    if ok:
        n_pass += 1
    print(f"  [{tag}] Identity {name}")
print("-" * 60)
print(f"  {n_pass}/{len(checks)} identities verified.")

SCORECARD: Thermal Dynamics on Z*_210
  [PASS] Identity 51: Heat trace factorizes
  [PASS] Identity 52: Triangle-free (both)
  [PASS] Identity 53: Z = (1+e^-2b)^3 * f1*f2
  [PASS] Identity 54: det'(L) = 2^25*3^16*5^13*7^8
  [PASS] Identity 55: All coupled eigs in Z[sqrt3]
------------------------------------------------------------
  5/5 identities verified.


## Summary

Five new identities (#51–#55) reveal the **thermal dynamics** of $\mathbb{Z}^*_{210}$:

1. **Heat trace factorization** — Under separable generators, diffusion in each faculty
   is independent. The return probability factors as $P_3 \cdot P_5 \cdot P_7$.

2. **Triangle-free** — No three generators multiply to the identity. The Cayley graph
   has girth 4, not 3. No closed 3-step walks exist.

3. **Closed-form partition function** — The primes 3 and 5 produce a perfect cube
   $(1+e^{-2\beta})^3$, while prime 7 contributes a $\sqrt{3}$ factorization.
   The thermal structure splits cleanly between equal-cost faculties (3,5) and the
   lowest-cost faculty (7).

4. **Spectral determinant** — Only the four primordial primes $\{2,3,5,7\}$ appear.
   The Kirchhoff spanning tree count is a pure product of $\{2,3,5,7\}$ powers.

5. **$\mathbb{Q}(\sqrt{3})$ universality** — The algebraic closure of the entire
   spectrum is determined by $\lambda(210) = 12$ through the real cyclotomic field
   $\mathbb{Q}(\cos\pi/6) = \mathbb{Q}(\sqrt{3})$. The same $\sqrt{3}$ governs
   the coupled gap, the partition function, and every irrational eigenvalue.

Running total: **55 structural identities, 0 free parameters, 2 honest nulls.**